In [1]:
# Cell 2
import pandas as pd
import requests
from tqdm.auto import tqdm

# Historical API (correct for past dates)
API_URL = "https://archive-api.open-meteo.com/v1/archive"

LAT = 28.6519
LON = 77.2315
TIMEOUT = 10

START_DATE = "2022-01-01"   # yyyy-mm-dd
END_DATE   = "2026-04-01"   # yyyy-mm-dd

OUT_1H = r"C:\Users\suhan\Desktop\ELECTRICITY_2026\JUPYTER NOTEBOOK\hourly_data.csv"
OUT_5M = r"C:\Users\suhan\Desktop\ELECTRICITY_2026\JUPYTER NOTEBOOK\Delhi_Weather_5M.csv"

HOURLY_PARAMS = ",".join([
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
])


In [3]:
# Cell 3
def fetch_chunk(start_date, end_date):
    params = {
        "latitude": LAT,
        "longitude": LON,
        "hourly": HOURLY_PARAMS,
        "timezone": "Asia/Kolkata",
        "start_date": start_date,
        "end_date": end_date,
    }
    r = requests.get(API_URL, params=params, timeout=TIMEOUT)
    r.raise_for_status()
    js = r.json()["hourly"]

    return pd.DataFrame({
        "datetime": pd.to_datetime(js["time"]),
        "Temperature (°C)": js["temperature_2m"],
        "Apparent Temperature (°C)": js["apparent_temperature"],
        "Relative Humidity (%)": js["relative_humidity_2m"],
        "Wind Speed (m/s)": js["wind_speed_10m"],
        "Precipitation (mm)": js["precipitation"],
        "Cloud Cover Total (%)": js["cloud_cover"],
        "Cloud Cover Low (%)": js["cloud_cover_low"],
        "Cloud Cover Mid (%)": js["cloud_cover_mid"],
        "Cloud Cover High (%)": js["cloud_cover_high"],
    })

def make_chunks(start, end, days=365):
    s = pd.Timestamp(start)
    e = pd.Timestamp(end)
    out = []
    while s <= e:
        t = min(s + pd.Timedelta(days=days-1), e)
        out.append((s.strftime("%Y-%m-%d"), t.strftime("%Y-%m-%d")))
        s = t + pd.Timedelta(days=1)
    return out


In [5]:
# Cell 4: Fetch hourly
parts = []
for s, e in tqdm(make_chunks(START_DATE, END_DATE), desc="Fetching hourly"):
    parts.append(fetch_chunk(s, e))

hourly = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates(subset=["datetime"])
    .sort_values("datetime")
    .reset_index(drop=True)
)

# Match your dataset format
hourly["Date"] = hourly["datetime"].dt.strftime("%d/%m/%Y")
hourly["TimeSlot"] = hourly["datetime"].dt.strftime("%H:%M")

hourly_out = hourly[
    ["Date","TimeSlot",
     "Temperature (°C)","Relative Humidity (%)","Apparent Temperature (°C)",
     "Precipitation (mm)","Wind Speed (m/s)",
     "Cloud Cover Total (%)","Cloud Cover Low (%)","Cloud Cover Mid (%)","Cloud Cover High (%)"]
]

hourly_out.to_csv(OUT_1H, index=False)
print("Saved hourly:", OUT_1H, "rows:", len(hourly_out))
hourly_out.head()


Fetching hourly:   0%|          | 0/5 [00:00<?, ?it/s]

Saved hourly: C:\Users\suhan\Desktop\ELECTRICITY_2026\JUPYTER NOTEBOOK\hourly_data.csv rows: 37248


,Date,TimeSlot,Temperature (°C),Relative Humidity (%),Apparent Temperature (°C),Precipitation (mm),Wind Speed (m/s),Cloud Cover Total (%),Cloud Cover Low (%),Cloud Cover Mid (%),Cloud Cover High (%)
0,01/01/2022,00:00,7.2,91,5.1,0.0,7.6,4,0,0,4
1,01/01/2022,01:00,6.9,92,4.8,0.0,7.1,0,0,0,0
2,01/01/2022,02:00,6.6,93,4.3,0.0,7.6,0,0,0,0
3,01/01/2022,03:00,6.4,93,4.3,0.0,6.8,0,0,0,0
4,01/01/2022,04:00,6.3,94,4.2,0.0,6.6,9,0,9,0


In [7]:
# Cell 5: Convert to 5-minute
five = hourly.set_index("datetime").sort_index()

full_idx = pd.date_range(five.index.min(), five.index.max(), freq="5min")
five = five.reindex(full_idx).interpolate(method="time", limit_direction="both")
five.index.name = "datetime"
five = five.reset_index()

five["Date"] = five["datetime"].dt.strftime("%d/%m/%Y")
five["TimeSlot"] = five["datetime"].dt.strftime("%H:%M")

five_out = five[
    ["Date","TimeSlot",
     "Temperature (°C)","Relative Humidity (%)","Apparent Temperature (°C)",
     "Precipitation (mm)","Wind Speed (m/s)",
     "Cloud Cover Total (%)","Cloud Cover Low (%)","Cloud Cover Mid (%)","Cloud Cover High (%)"]
]

five_out.to_csv(OUT_5M, index=False)
print("Saved 5-minute:", OUT_5M, "rows:", len(five_out))
five_out.head()


C:\Users\suhan\AppData\Local\Temp\ipykernel_13372\2885234073.py:5: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  five = five.reindex(full_idx).interpolate(method="time", limit_direction="both")


Saved 5-minute: C:\Users\suhan\Desktop\ELECTRICITY_2026\JUPYTER NOTEBOOK\Delhi_Weather_5M.csv rows: 446965


,Date,TimeSlot,Temperature (°C),Relative Humidity (%),Apparent Temperature (°C),Precipitation (mm),Wind Speed (m/s),Cloud Cover Total (%),Cloud Cover Low (%),Cloud Cover Mid (%),Cloud Cover High (%)
0,01/01/2022,00:00,7.200,91.000000,5.100,0.0,7.600000,4.000000,0.0,0.0,4.000000
1,01/01/2022,00:05,7.175,91.083333,5.075,0.0,7.558333,3.666667,0.0,0.0,3.666667
2,01/01/2022,00:10,7.150,91.166667,5.050,0.0,7.516667,3.333333,0.0,0.0,3.333333
3,01/01/2022,00:15,7.125,91.250000,5.025,0.0,7.475000,3.000000,0.0,0.0,3.000000
4,01/01/2022,00:20,7.100,91.333333,5.000,0.0,7.433333,2.666667,0.0,0.0,2.666667


In [9]:
# Cell 6: Quick checks
print("Hourly nulls:\n", hourly_out.isna().sum())
print("\n5M nulls:\n", five_out.isna().sum())

tmp = five_out.copy()
tmp["dt"] = pd.to_datetime(tmp["Date"] + " " + tmp["TimeSlot"], format="%d/%m/%Y %H:%M")
tmp = tmp.sort_values("dt")
print("\nCommon gap minutes:", tmp["dt"].diff().dt.total_seconds().div(60).dropna().mode().tolist()[:3])


Hourly nulls:
 Date                         0
TimeSlot                     0
Temperature (°C)             0
Relative Humidity (%)        0
Apparent Temperature (°C)    0
Precipitation (mm)           0
Wind Speed (m/s)             0
Cloud Cover Total (%)        0
Cloud Cover Low (%)          0
Cloud Cover Mid (%)          0
Cloud Cover High (%)         0
dtype: int64

5M nulls:
 Date                         0
TimeSlot                     0
Temperature (°C)             0
Relative Humidity (%)        0
Apparent Temperature (°C)    0
Precipitation (mm)           0
Wind Speed (m/s)             0
Cloud Cover Total (%)        0
Cloud Cover Low (%)          0
Cloud Cover Mid (%)          0
Cloud Cover High (%)         0
dtype: int64

Common gap minutes: [5.0]
